In [14]:
import re
import pandas as pd
import dotenv
import os
from neo4j import GraphDatabase

In [92]:
triplets_df_path = "../data/TRIPLETS_ALL_prenorm.csv"
env_file = "../Neo4j_private.txt"

# clear neo4j 

In [93]:
dotenv.load_dotenv(env_file)

True

In [94]:
from neo4j import GraphDatabase

dotenv.load_dotenv(env_file)
URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PWD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(URI, auth=(USER, PWD))

with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

driver.close()

# ------

In [79]:
df = pd.read_csv(triplets_df_path)

In [81]:
#df.loc[
#    (df['entity1_type'] == 'country')]["entity1"].unique()

In [88]:
df.loc[
    (df['entity1_type'] == 'country') |
    (df['entity2_type'] == 'country')
] 

,entity1,entity1_type,rel_type,entity2,entity2_type,sector,url,date
14,First Iraqi Bank,company,is_member_of,Iraq,country,financials,https://gfmag.com/executive-interviews/complia...,2025-12-30T15:24:25+00:00
16,First Iraqi Bank,company,has_positive_impact,Iraq,country,financials,https://gfmag.com/executive-interviews/complia...,2025-12-30T15:24:25+00:00
18,First Iraqi Bank,company,has_positive_impact,Iraq,country,financials,https://gfmag.com/executive-interviews/complia...,2025-12-30T15:24:25+00:00
20,First Iraqi Bank,company,has_positive_impact,Iraq,country,financials,https://gfmag.com/executive-interviews/complia...,2025-12-30T15:24:25+00:00
28,Financial Action Task Force,regulator,controls,South Africa,country,financials,https://gfmag.com/news/fatf-removes-4-countrie...,2025-12-19T17:47:20+00:00
...,...,...,...,...,...,...,...,...
30811,Singapore,country,is_member_of,Asean economic region,economic_indicator,public_sector,https://gfmag.com/data/shifting-fortunes/,2017-07-21T00:00:00+00:00
30812,Singapore,country,is_member_of,Association of Southeast Asian Nations,economic_indicator,public_sector,https://gfmag.com/data/shifting-fortunes/,2017-07-21T00:00:00+00:00
30813,Singapore,country,is_member_of,Asean economic region,economic_indicator,public_sector,https://gfmag.com/data/shifting-fortunes/,2017-07-21T00:00:00+00:00
30814,Singapore,country,is_member_of,Association of Southeast Asian Nations,economic_indicator,public_sector,https://gfmag.com/data/shifting-fortunes/,2017-07-21T00:00:00+00:00


In [83]:
import pycountry

In [56]:
pycountry.countries.lookup("AI").name

'Anguilla'

In [73]:
def normalize_countries(name: str, entity_type: str):
    """
    Conservative country normalization.
    Never upgrades non-country entities.
    """
    cleaned = re.sub(r"[^\w\s]", "", name)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    if entity_type != "country":
        return cleaned, entity_type
    try:
        country = pycountry.countries.lookup(cleaned)
        return country.name, "country"
    except:
        return cleaned, "other"

In [72]:
e1_name, e1_type

('Nicaragua', 'country')

In [63]:
country = pycountry.countries.lookup("AI")

In [64]:
country

Country(alpha_2='AI', alpha_3='AIA', flag='🇦🇮', name='Anguilla', numeric='660')

In [65]:
country.name

'Anguilla'

In [86]:
df.dropna(subset=["entity1", "entity2"], inplace=True)

In [87]:
df[["entity1", "entity1_type"]] = df.apply(
    lambda row: normalize_countries(row["entity1"], row["entity1_type"]),
    axis=1,
    result_type="expand"
)
df[["entity2", "entity2_type"]] = df.apply(
    lambda row: normalize_countries(row["entity2"], row["entity2_type"]),
    axis=1,
    result_type="expand"
)

In [90]:
df.to_csv("../data/TRIPLETS_ALL.csv", index=False)

In [95]:
df.loc[df["rel_type"]=="invests_in"]

,entity1,entity1_type,rel_type,entity2,entity2_type,sector,url,date
10,corporates,company,invests_in,digital tools,product_service,technology,https://gfmag.com/executive-interviews/standar...,2025-12-31T18:00:00+00:00
22,Mirae Asset Securities,company,invests_in,artificial intelligence,product_service,technology,https://gfmag.com/banking/mirae-asset-securiti...,2025-12-16T11:02:48+00:00
58,Qatar Investment Authority,financial_institution,invests_in,Anthropic,company,technology,https://gfmag.com/features/qatar-new-world-cap...,2025-12-10T20:41:50+00:00
59,Qatar Investment Authority,financial_institution,invests_in,Applied Intuition,company,technology,https://gfmag.com/features/qatar-new-world-cap...,2025-12-10T20:41:50+00:00
181,Banks,financial_institution,invests_in,Private credit,financial_instrument,financials,https://gfmag.com/banking/familiar-bedfellows/,2025-10-08T15:04:07+00:00
...,...,...,...,...,...,...,...,...
30624,FAB,financial_institution,invests_in,cash management and liquidity solutions,product_service,financials,https://gfmag.com/award/best-treasury-cash-man...,2024-07-26T15:37:10+00:00
30720,Citi,financial_institution,invests_in,cloudbased flexible microservices,product_service,technology,https://gfmag.com/executive-interviews/reaping...,2023-09-03T00:00:00+00:00
30772,Nordea Bank,financial_institution,invests_in,blockchain,financial_instrument,technology,https://gfmag.com/executive-interviews/nordea-...,2019-12-09T00:00:00+00:00
30790,Data Collective venture capital fund,financial_institution,invests_in,Merlon Intelligence,company,financials,https://gfmag.com/data/can-technology-ensure-c...,2017-07-21T00:00:00+00:00
